# Week 2 — Features and validation (for the group)

Hi team — Week 1 gave us a clean monthly table with **real** PM2.5 only. Here we turn that into something sklearn can eat: **lags**, **season signals**, and two kinds of **train/test split** (time + space).

**Why two splits?** A random split lets the model cheat (same station in train and test). **Time split** = “can we predict future months?” **Spatial split** = “can we predict stations the model never saw?”

**Input:** `datasets/week1_openaq_base.csv` from `week1_clean_monthly_no_annual_fill.ipynb`.

**Output:** `datasets/week2_features.csv` for Week 3.


### Imports

pandas, numpy, and `GroupShuffleSplit` (keeps whole stations together in the spatial test). Use your project env (e.g. `pm25`) if anything fails to import.


In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit

pd.set_option("display.max_columns", 200)
print("Libraries loaded")


Libraries loaded


### Read the Week 1 CSV

Handoff from Week 1. You should see `openaq_id`, `date`, `PM2_5`, plus weather/context. If the row count is tiny, rerun Week 1 first.


In [15]:
ROOT = Path(".")
INPUT = ROOT / "datasets" / "week1_openaq_base.csv"
OUTPUT = ROOT / "datasets" / "week2_features.csv"

df = pd.read_csv(INPUT, parse_dates=["date"])
print("Loaded:", INPUT)
print("Shape:", df.shape)
display(df.head())


Loaded: datasets/week1_openaq_base.csv
Shape: (4151, 9)


,openaq_id,Year,Month,date,Season,PM2_5,PM10,NO2,O3
0,7498,2020,2,2020-02-01,Winter,25.70,29.3,27.8,47.3
1,7498,2020,3,2020-03-01,Spring,15.30,19.7,NaN,71.5
2,7498,2020,4,2020-04-01,Spring,8.17,12.3,NaN,60.8
3,7512,2019,12,2019-12-01,Winter,11.00,18.4,15.9,NaN
4,7512,2020,1,2020-01-01,Winter,8.50,17.9,13.7,NaN


### Sort by station, then time

**Must-do before lags:** `shift()` only works if each station’s rows are in chronological order. We sort by `openaq_id` and `date`, then `reset_index(drop=True)`.


In [16]:
df = df.sort_values(["openaq_id", "date"]).reset_index(drop=True)
display(df[["openaq_id", "date", "PM2_5"]].head(10))


,openaq_id,date,PM2_5
0,7498,2020-02-01,25.70
1,7498,2020-03-01,15.30
2,7498,2020-04-01,8.17
3,7512,2019-12-01,11.00
4,7512,2020-01-01,8.50
5,7512,2020-02-01,9.68
6,7512,2021-10-01,7.06
7,7512,2021-11-01,5.81
8,7512,2022-08-01,6.00
9,7512,2022-09-01,10.60


### Calendar / season features

Month index plus **sin/cos** so December is “near” January in a smooth way. Trees can learn season without this, but it helps Ridge and costs almost nothing.


In [17]:
df["month_num"] = df["Month"]
df["month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)
display(df[["Month", "month_num", "month_sin", "month_cos"]].head())


,Month,month_num,month_sin,month_cos
0,2,2,8.660254e-01,5.000000e-01
1,3,3,1.000000e+00,6.123234e-17
2,4,4,8.660254e-01,-5.000000e-01
3,12,12,-2.449294e-16,1.000000e+00
4,1,1,5.000000e-01,8.660254e-01


### PM2.5 lags and short rolling stats

We use **past** months only (`shift(1)`, rolling on shifted series) so we don’t leak the current month’s PM2.5 into the features. **Tell the group:** if someone uses same-month PM2.5 as an input feature, that’s cheating.

Lags are usually strong here — that’s normal for monthly air data.


In [18]:
group = df.groupby("openaq_id", group_keys=False)
df["PM2_5_lag1"] = group["PM2_5"].shift(1)
df["PM2_5_lag2"] = group["PM2_5"].shift(2)
df["PM2_5_lag3"] = group["PM2_5"].shift(3)
df["PM2_5_roll3_mean"] = group["PM2_5"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
df["PM2_5_roll3_std"] = group["PM2_5"].shift(1).rolling(3).std().reset_index(level=0, drop=True)
display(df[["openaq_id", "date", "PM2_5", "PM2_5_lag1", "PM2_5_lag2", "PM2_5_lag3", "PM2_5_roll3_mean"]].head(12))


,openaq_id,date,PM2_5,PM2_5_lag1,PM2_5_lag2,PM2_5_lag3,PM2_5_roll3_mean
0,7498,2020-02-01,25.70,NaN,NaN,NaN,NaN
1,7498,2020-03-01,15.30,25.70,NaN,NaN,NaN
2,7498,2020-04-01,8.17,15.30,25.70,NaN,NaN
3,7512,2019-12-01,11.00,NaN,NaN,NaN,NaN
4,7512,2020-01-01,8.50,11.00,NaN,NaN,NaN
5,7512,2020-02-01,9.68,8.50,11.00,NaN,NaN
6,7512,2021-10-01,7.06,9.68,8.50,11.00,9.726667
7,7512,2021-11-01,5.81,7.06,9.68,8.50,8.413333
8,7512,2022-08-01,6.00,5.81,7.06,9.68,7.516667
9,7512,2022-09-01,10.60,6.00,5.81,7.06,6.290000


### Weather lag (previous month)

`Temp_Mean_lag1`, etc. tie this month’s target to **last month’s** weather. Missing values can happen; Week 3’s pipeline imputes.


In [19]:
for c in ["Temp_Mean", "Wind_Speed", "Precipitation"]:
    if c in df.columns:
        df[f"{c}_lag1"] = group[c].shift(1)

weather_lag_cols = [c for c in df.columns if c.endswith("_lag1") and ("Temp" in c or "Wind" in c or "Precipitation" in c)]
print("Weather lag columns:", weather_lag_cols)
display(df[["openaq_id", "date"] + weather_lag_cols].head(10))


Weather lag columns: []


,openaq_id,date
0,7498,2020-02-01
1,7498,2020-03-01
2,7498,2020-04-01
3,7512,2019-12-01
4,7512,2020-01-01
5,7512,2020-02-01
6,7512,2021-10-01
7,7512,2021-11-01
8,7512,2022-08-01
9,7512,2022-09-01


### Other pollutants lag1 (optional)

PM10 / NO2 / O3 from the **previous** month as extra signal. Lots of NaNs is OK for predictors; the target must stay clean.


In [20]:
for c in ["PM10", "NO2", "O3"]:
    if c in df.columns:
        df[f"{c}_lag1"] = group[c].shift(1)

poll_lag_cols = [c for c in ["PM10_lag1", "NO2_lag1", "O3_lag1"] if c in df.columns]
print("Pollutant lag columns:", poll_lag_cols)
display(df[["openaq_id", "date"] + poll_lag_cols].head(10))


Pollutant lag columns: ['PM10_lag1', 'NO2_lag1', 'O3_lag1']


,openaq_id,date,PM10_lag1,NO2_lag1,O3_lag1
0,7498,2020-02-01,NaN,NaN,NaN
1,7498,2020-03-01,29.3,27.8,47.3
2,7498,2020-04-01,19.7,NaN,71.5
3,7512,2019-12-01,NaN,NaN,NaN
4,7512,2020-01-01,18.4,15.9,NaN
5,7512,2020-02-01,17.9,13.7,NaN
6,7512,2021-10-01,19.5,11.2,NaN
7,7512,2021-11-01,14.8,16.2,NaN
8,7512,2022-08-01,11.0,16.9,NaN
9,7512,2022-09-01,16.0,19.6,NaN


### Drop rows without enough history

The first months per station don’t have3 lags yet — we drop them. Row count **drops** vs Week 1; that’s expected.


In [21]:
required_lags = ["PM2_5_lag1", "PM2_5_lag2", "PM2_5_lag3"]
df_model = df.dropna(subset=required_lags).copy()
print("Rows before:", len(df))
print("Rows after lag filter:", len(df_model))
display(df_model.head())


Rows before: 4151
Rows after lag filter: 3719


,openaq_id,Year,Month,date,Season,PM2_5,PM10,NO2,O3,month_num,month_sin,month_cos,PM2_5_lag1,PM2_5_lag2,PM2_5_lag3,PM2_5_roll3_mean,PM2_5_roll3_std,PM10_lag1,NO2_lag1,O3_lag1
6,7512,2021,10,2021-10-01,Autumn,7.06,14.8,16.2,NaN,10,-0.866025,5.000000e-01,9.68,8.50,11.00,9.726667,1.250653,19.5,11.2,NaN
7,7512,2021,11,2021-11-01,Autumn,5.81,11.0,16.9,NaN,11,-0.500000,8.660254e-01,7.06,9.68,8.50,8.413333,1.312148,14.8,16.2,NaN
8,7512,2022,8,2022-08-01,Summer,6.00,16.0,19.6,NaN,8,-0.866025,-5.000000e-01,5.81,7.06,9.68,7.516667,1.975002,11.0,16.9,NaN
9,7512,2022,9,2022-09-01,Autumn,10.60,24.2,14.9,NaN,9,-1.000000,-1.836970e-16,6.00,5.81,7.06,6.290000,0.673573,16.0,19.6,NaN
10,7512,2022,10,2022-10-01,Autumn,6.97,13.7,13.2,NaN,10,-0.866025,5.000000e-01,10.60,6.00,5.81,7.470000,2.712324,24.2,14.9,NaN


### Missing values after features

Quick sanity check before splits. If half the table is NaN, we probably broke a column name or merge.


In [22]:
missing_pct = (df_model.isna().mean() * 100).sort_values(ascending=False).round(2)
print("Missing % after feature creation:")
print(missing_pct.head(25))


Missing % after feature creation:
O3                  46.68
O3_lag1             46.63
PM10                15.03
PM10_lag1           15.00
NO2_lag1             2.15
NO2                  2.15
PM2_5_lag1           0.00
PM2_5_roll3_std      0.00
PM2_5_roll3_mean     0.00
PM2_5_lag3           0.00
PM2_5_lag2           0.00
openaq_id            0.00
month_cos            0.00
Year                 0.00
month_num            0.00
PM2_5                0.00
Season               0.00
date                 0.00
Month                0.00
month_sin            0.00
dtype: float64


### Time split (train on the past, test on the future)

Cutoff = **80th percentile** of `date` (roughly last ~20% of timeline in test). Good default for a class project; you can switch to a fixed calendar year for the report.

**This is the split to stress** for “forecasting” language.


In [23]:
cutoff_date = df_model["date"].quantile(0.8)
train_time = df_model[df_model["date"] <= cutoff_date].copy()
test_time = df_model[df_model["date"] > cutoff_date].copy()

print("Cutoff date:", cutoff_date.date())
print("Train rows:", len(train_time))
print("Test rows:", len(test_time))
print("Train date range:", train_time["date"].min().date(), "to", train_time["date"].max().date())
print("Test date range:", test_time["date"].min().date(), "to", test_time["date"].max().date())


Cutoff date: 2025-04-01
Train rows: 3012
Test rows: 707
Train date range: 2020-03-01 to 2025-04-01
Test date range: 2025-05-01 to 2025-11-01


### Spatial split (hide entire stations)

`GroupShuffleSplit` puts every row of a station on **one** side only. If spatial scores are much worse than time scores, that’s a **result** (generalization is hard), not necessarily a bug.


In [24]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_train, idx_test = next(splitter.split(df_model, groups=df_model["openaq_id"]))
train_space = df_model.iloc[idx_train].copy()
test_space = df_model.iloc[idx_test].copy()

print("Spatial train rows:", len(train_space))
print("Spatial test rows:", len(test_space))
print("Train stations:", train_space["openaq_id"].nunique())
print("Test stations:", test_space["openaq_id"].nunique())


Spatial train rows: 2945
Spatial test rows: 774
Train stations: 91
Test stations: 23


### Feature list for Week 3

Only columns that exist get passed through. Ridge + Random Forest in Week 3 use this list + target `PM2_5`.


In [25]:
candidate_features = [
    "PM2_5_lag1", "PM2_5_lag2", "PM2_5_lag3", "PM2_5_roll3_mean", "PM2_5_roll3_std",
    "PM10_lag1", "NO2_lag1", "O3_lag1",
    "Temp_Mean_lag1", "Wind_Speed_lag1", "Precipitation_lag1",
    "month_num", "month_sin", "month_cos",
    "Green_Ratio", "Latitude", "Longitude"
]

feature_cols = [c for c in candidate_features if c in df_model.columns]
print("Feature count:", len(feature_cols))
print(feature_cols)


Feature count: 11
['PM2_5_lag1', 'PM2_5_lag2', 'PM2_5_lag3', 'PM2_5_roll3_mean', 'PM2_5_roll3_std', 'PM10_lag1', 'NO2_lag1', 'O3_lag1', 'month_num', 'month_sin', 'month_cos']


### Save `week2_features.csv`

Regenerate this file after changing Week 1 — avoid hand-editing in Excel (decimals/encoding).


In [26]:
save_cols = ["openaq_id", "date", "Year", "Month", "PM2_5"] + feature_cols
save_cols = [c for c in save_cols if c in df_model.columns]

df_model[save_cols].to_csv(OUTPUT, index=False)
print("Saved:", OUTPUT)
print("Final shape:", df_model[save_cols].shape)
display(df_model[save_cols].head())


Saved: datasets/week2_features.csv
Final shape: (3719, 16)


,openaq_id,date,Year,Month,PM2_5,PM2_5_lag1,PM2_5_lag2,PM2_5_lag3,PM2_5_roll3_mean,PM2_5_roll3_std,PM10_lag1,NO2_lag1,O3_lag1,month_num,month_sin,month_cos
6,7512,2021-10-01,2021,10,7.06,9.68,8.50,11.00,9.726667,1.250653,19.5,11.2,NaN,10,-0.866025,5.000000e-01
7,7512,2021-11-01,2021,11,5.81,7.06,9.68,8.50,8.413333,1.312148,14.8,16.2,NaN,11,-0.500000,8.660254e-01
8,7512,2022-08-01,2022,8,6.00,5.81,7.06,9.68,7.516667,1.975002,11.0,16.9,NaN,8,-0.866025,-5.000000e-01
9,7512,2022-09-01,2022,9,10.60,6.00,5.81,7.06,6.290000,0.673573,16.0,19.6,NaN,9,-1.000000,-1.836970e-16
10,7512,2022-10-01,2022,10,6.97,10.60,6.00,5.81,7.470000,2.712324,24.2,14.9,NaN,10,-0.866025,5.000000e-01


### Next: Week 3

Open **`week3_modeling_and_shap.ipynb`**: train Ridge & Random Forest, plots, SHAP, and a printed **champion** model.
